In [1]:
import pandas as pd
import pickle
import numpy as np

In [2]:
# 1. Load Model
with open('../../models/model_progress.pickle', 'rb') as f:
    data_model = pickle.load(f)

models = data_model['models_dict']
encoders = data_model['encoders']
feature_order = data_model['features']

In [3]:
# --- FUNGSI PREDIKSI ---
def prediksi_progres_lengkap(user_data, minggu_ke):
    # A. Hitung Data Turunan & B. Encoding (Sama seperti sebelumnya)
    tinggi_m = user_data['Height_cm'] / 100
    bmi_awal = round(user_data['Initial_Weight_kg'] / (tinggi_m ** 2), 2)
    
    # Hitung kategori BMI awal kasar
    if bmi_awal < 18.5: cat_bmi = 'Underweight'
    elif bmi_awal < 25: cat_bmi = 'Normal'
    elif bmi_awal < 30: cat_bmi = 'Overweight'
    else: cat_bmi = 'Obese'
    
    gender_code = encoders['Gender'].transform([user_data['Gender']])[0]
    goal_code = encoders['Goal'].transform([user_data['Goal']])[0]
    level_code = encoders['level'].transform([user_data['level']])[0]
    bmi_cat_code = encoders['BMI_Category_x'].transform([cat_bmi])[0]

    # C. Susun Array Input
    input_row = [
        user_data['Age'], 
        gender_code, 
        user_data['Height_cm'], 
        user_data['Initial_Weight_kg'],
        bmi_awal, 
        bmi_cat_code, 
        user_data['Body_Fat_Category'], 
        user_data['Body_Fat_Percentage_x'],
        goal_code, 
        user_data['Workout_Frequency'], 
        user_data['Average_Duration_Minutes'], 
        level_code,
        user_data['Badminton'], 
        user_data['Football'], 
        user_data['Basketball'],
        user_data['Tennis'], 
        user_data['Volleyball'], 
        user_data['Table_Tennis'], 
        user_data['Swim'],
        minggu_ke
    ]
    
    # Bungkus jadi DataFrame
    input_df = pd.DataFrame([input_row], columns=feature_order)
    
    # D. Lakukan Prediksi UNTUK SEMUA MODEL
    hasil = {}
    
    # Kita loop semua model yang ada di dictionary 'models'
    # target_name = nama kolom (misal: Weight_kg, Daily_Calories, Progress_Status_Encoded)
    # info = isinya {'model': ..., 'type': ...}
    for target_name, info in models.items():
        
        # Prediksi nilainya
        nilai_prediksi = info['model'].predict(input_df)[0]
        
        # Cek tipe model (Angka atau Kategori?)
        if info['type'] == 'categorical':
            # Kalau kategori, kita harus terjemahkan balik (Decode)
            # Nama encoder biasanya nama target tanpa "_Encoded"
            nama_asli = target_name.replace('_Encoded', '')
            
            # Terjemahkan angka ke teks
            teks = encoders[nama_asli].inverse_transform([int(nilai_prediksi)])[0]
            hasil[nama_asli] = teks
        else:
            # Kalau angka, langsung simpan
            hasil[target_name] = nilai_prediksi
            
    return hasil


In [4]:
input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Initial_Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Body_Fat_Percentage_x': 15.0,
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Tennis': 0,
    'Volleyball': 0, 
    'Table_Tennis': 0, 
    'Swim': 0
}

print(f"User: Pria, 25th, 60kg -> Goal: Muscle Gain")

for minggu in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    output = prediksi_progres_lengkap(input_user, minggu_ke=minggu)
    
    print(f"\n{'='*10} MINGGU KE-{minggu} {'='*10}")
    
    # Kelompokkan output biar enak dibaca
    print(f"[FISIK]")
    print(f"  Berat Badan     : {output['Weight_kg']:.2f} kg")
    print(f"  BMI             : {output['BMI']:.2f} ({output['BMI_Category_y']})")
    print(f"  Body Fat        : {output['Body_Fat_Percentage_y']:.1f}%")
    
    print(f"[NUTRISI HARIAN]")
    print(f"  Kalori          : {output['Daily_Calories']:.0f} kkal")
    print(f"  Air Minum       : {output['Daily_Water_ml']:.0f} ml")
    print(f"  Gula (Limit)    : {output['Limit_Sugar_g']:.1f} g")
    
    print(f"[MAKRO NUTRISI]")
    print(f"  Protein         : {output['Target_Protein_g']:.1f} g")
    print(f"  Karbo           : {output['Target_Carbs_g']:.1f} g")
    print(f"  Lemak           : {output['Target_Fat_g']:.1f} g")
    print(f"  Serat           : {output['Target_Fiber_g']:.1f} g")
    print(f"  Kalsium         : {output['Target_Calcium_mg']:.0f} mg")
    print(f"  Kolesterol (Max): {output['Limit_Cholesterol_mg']:.0f} mg")

User: Pria, 25th, 60kg -> Goal: Muscle Gain

========== MINGGU KE-1 ==========
[FISIK]
  Berat Badan     : 60.04 kg
  BMI             : 19.22 (Normal)
  Body Fat        : 14.9%
[NUTRISI HARIAN]
  Kalori          : 2738 kkal
  Air Minum       : 2788 ml
  Gula (Limit)    : 68.0 g
[MAKRO NUTRISI]
  Protein         : 208.2 g
  Karbo           : 332.7 g
  Lemak           : 59.0 g
  Serat           : 37.8 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-2 ==========
[FISIK]
  Berat Badan     : 60.20 kg
  BMI             : 19.27 (Normal)
  Body Fat        : 14.9%
[NUTRISI HARIAN]
  Kalori          : 2738 kkal
  Air Minum       : 2788 ml
  Gula (Limit)    : 68.0 g
[MAKRO NUTRISI]
  Protein         : 208.2 g
  Karbo           : 332.7 g
  Lemak           : 59.0 g
  Serat           : 37.8 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-3 ==========
[FISIK]
  Berat Badan     : 60.30 kg
  BMI             : 19.32 (Normal)
  Body Fat       